# Pune Property Price - EDA (v2)
**Setup:** keep `eda_utils.py`, `DOC-20261110-WA0011.xlsx` and this notebook in the same folder, then run all cells.
`pip install -r requirements.txt` if a library is missing.

**What changed vs v1:** reusable functions in `eda_utils.py`, relative file path with a clear error message, BOM cleaning inside `load_data`,
log-transform and capping for skewed price per sqft, VIF after fixing multicollinearity, and two extra tests
(Property Type after controlling for Area; how much Area + Location explain on their own).

## 1. Load and prepare

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from eda_utils import *

pd.set_option("display.width", 140)
raw = load_data()          # handles the hidden BOM in 'Property ID' and renames columns
df = add_features(raw)     # Price_Cr, PPSF, log_price, log_ppsf, PPSF_capped, buckets
df.head()

## 2. Data quality

In [ ]:
print(pd.Series(quality_report(df)))
df.dtypes

No missing values or duplicates. The 71 rows with more bathrooms than bedrooms and the Independent Houses above ground floor are flagged, not removed.

In [ ]:
df[NUM_COLS].describe().T

## 3. Target: price and its transform

In [ ]:
print('Price skew:', round(df.Price.skew(), 2), ' log(Price) skew:', round(df.log_price.skew(), 2))
plot_price_distribution(df); plt.show()

## 4. Feature distributions

In [ ]:
plot_numeric_distributions(df); plt.show()
plot_categorical_counts(df); plt.show()

## 5. Price vs categorical features

In [ ]:
plot_price_by_category(df); plt.show()
group_tests(df, target='Price')

## 6. Price vs numeric features and correlation

In [ ]:
plot_correlation(df); plt.show()
plot_scatter(df); plt.show()
df[NUM_COLS].corr()['Price'].drop('Price').sort_values(ascending=False).round(3)

## 7. Multicollinearity (VIF) and the fix

In [ ]:
base = ['Area', 'Bedrooms', 'Bathrooms', 'Floor', 'Age', 'Distance']
fixed = ['Area', 'Bath_Bed_Ratio', 'Floor', 'Age', 'Distance']
pd.concat([vif_table(df, base).rename('VIF original'),
           vif_table(df, fixed).rename('VIF after fix')], axis=1)

Bedrooms and Bathrooms overlap with Area (VIF 7.5 and 5.1). Replacing them with `Bath_Bed_Ratio` (or dropping them) brings every VIF to about 1.

## 8. Price per sqft by location: skew, log transform and capping

In [ ]:
plot_ppsf_by_location(df); plt.show()
skew_table(df)

Skew is mild overall (0.48) and highest in Pimpri, Kharadi, Dhankawadi and Hadapsar (about 0.9-1.0). `log_ppsf` brings skew to about 0 overall; `PPSF_capped` (1st-99th percentile) is available if a model needs it.

In [ ]:
plot_location_type_heatmap(df); plt.show()
df.groupby('Location')[['PPSF', 'Price_Cr']].median().sort_values('PPSF', ascending=False).round(2)

## 9. Property Type: does it matter once Area is accounted for?

In [ ]:
print(type_effect_after_area(df))
df.groupby('Type').agg(median_area=('Area', 'median'), median_ppsf=('PPSF', 'median'), n=('Price', 'size'))

Villas are smaller on average but cost more per sqft, so the raw price comparison hides the type effect. After removing the effect of Area, Type is highly significant.

## 10. Age, bedrooms and outliers

In [ ]:
plot_age_bedrooms(df); plt.show()
plot_outliers(df); plt.show()
print('IQR outliers - Price:', int(iqr_outliers(df.Price).sum()),
      '| PPSF:', int(iqr_outliers(df.PPSF).sum()),
      '| log PPSF:', int(iqr_outliers(df.log_ppsf).sum()))

## 11. How much do Area and Location explain? (baseline linear model on log price)

In [ ]:
baseline_r2(df)

Area alone gives R2 of about 0.79 and Area + Location about 0.91; adding Type and Age raises it to about 0.93, and all features to about 0.94.

## 12. Summary and limitations
- Clean data: no missing values, no duplicates; hidden BOM in the first column header is removed in `load_data`.
- Area (r = 0.86) and Location are the main price drivers; Type and Age add a smaller but real effect.
- Floor, Distance, Furnishing and Parking show no meaningful effect on price.
- Bedrooms and Bathrooms are collinear with Area; use `Bath_Bed_Ratio` or drop them.
- Some patterns are unusual (houses on upper floors, no distance effect), so the data may be synthetic. This is an inference, not a proven fact.
- Cross-sectional data: no dates or interventions, so it cannot support synthetic control without a time dimension.